# MOHID Preprocessing

- Create regular grids
- Download, load and filter coastlines for grid region
- Perform interpolation on bathymetric data
- Update griddata depth values
- Convert the griddata to a shapefile
- Plot bathymetry

***
**Note 1**: Execute each cell through the <button class="btn btn-default btn-xs"><i class="icon-play fa fa-play"></i></button> button from the top MENU (or keyboard shortcut `Shift` + `Enter`).<br>
<br>
**Note 2**: Use the Kernel and Cell menus to restart the kernel and clear outputs.<br>
***

# Table of contents
- [1. Import required libraries](#1.-Import-required-libraries)
- [2. Load the XYZ data](#2.-Load-the-XYZ-data)
- [3. Grid](#3.-Get)
    - [3.1 Load a previously generated Mohid grid file](#3.1-Load-a-previously-generated-Mohid-grid-file)
    - [3.2 Create a new grid](#3.2-Create-a-new-grid)
        - [3.2.1 Get grid dimensions and spacing](#3.2.1-Get-grid-dimensions-and-spacing)
        - [3.2.2 Grid generation](#3.2.2-Grid-generation)
        - [3.2.3 Save the grid to a MOHID-compatible file](#3.2.3-Save-the-grid-to-a-MOHID-compatible-file) 
- [4. Coastline](#4.-Coastline)
    - [4.1 Load coastline](#4.1-Load-coastline)
        - [4.1.1 Load your coastline](#4.1.1-Load-your-coastline)
        - [4.1.2 GSHHG Coastline Data](#4.1.2-GSHHG-Coastline-Data)
        - [4.1.3 OSM Coastline Data](#4.1.3-OSM-Coastline-Data)
    - [4.2 Filter Coastlines for Grid Region](#4.2-Filter-Coastlines-for-Grid-Region)
    - [4.3 Show coastline](#4.3-Show-coastline)
- [5. Bathymetry](#5.-Bathymetry)
    - [5.1 Load a previously generated Mohid griddata file](#5.1-Load-a-previously-generated-Mohid-griddata-file)
    - [5.2 Create a new griddata](#5.2-Create-a-new-griddata)
        - [5.2.1 Download GEBCO_2026 bathymetry for the grid area (optional)](#5.2.1-Download-GEBCO_2026-bathymetry-for-the-grid-area-(optional))
        - [5.2.2 Interpolate bathymetry onto the grid](#5.2.2-Interpolate-bathymetry-onto-the-grid)
- [6. Visualize and update depth values by clicking on the map](#6.-Visualize-and-update-depth-values-by-clicking-on-the-map)
- [7. Save the griddata to a MOHID-compatible file](#7.-Save-the-griddata-to-a-MOHID-compatible-file)
- [8. Convert the griddata to a shapefile](#8.-Convert-the-griddata-to-a-shapefile)
- [9. Save shapefile to MOHID griddata](#9.-Save-shapefile-to-MOHID-griddata)
- [10. Plot MOHID griddata](#10.-Plot-MOHID-griddata)

# 1. Import required libraries

In [2]:
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from ipyleaflet import Map, Marker, basemaps, Popup, Polyline, Circle, GeoData, Polygon, GeoJSON, LayerGroup
from ipywidgets import HTML
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize, to_hex
import random
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter, label
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
from urllib.request import urlopen, Request
import io
from io import StringIO
from PIL import Image
import geopandas as gpd
from shapely.geometry import Point, box, Polygon
import requests
import zipfile
import os
from scipy.ndimage import label, find_objects, gaussian_filter
import matplotlib.colors as mcolors
import ipywidgets as widgets
import shapefile
from shapely import contains_xy
from shapely.ops import unary_union
from mpl_toolkits.axes_grid1 import make_axes_locatable
import time
import fiona
import pyogrio
from pathlib import Path
from math import radians, cos, sin
import math

# 2. Load the XYZ data
This step is optional (only perform if you already have an xyz file with bathymetry data).
If you already have a previously generated Mohid griddata file, go to [5.1 Load a previously generated Mohid griddata file](#5.1-Load-a-previously-generated-Mohid-griddata-file)

If you don't have bathymetry data, skip this step and download it from GEBCO_2026 in [5.2.1 Download GEBCO_2026 bathymetry for the grid area (optional)](#5.2.1-Download-GEBCO_2026-bathymetry-for-the-grid-area-(optional)).

In [ ]:
# 1. Configuration
file_path = Path(r'Data/data.xyz')  # Replace with your file path

MAX_POINTS = 1000  # maximum number of circles to draw

# 2. Read raw lines
raw_lines = file_path.read_text().splitlines()

# 3. Detect markers
has_begin = any(line.strip().startswith("<begin_xyz>") for line in raw_lines)
has_end   = any(line.strip().startswith("<end_xyz>")   for line in raw_lines)

# 4. Extract only the data lines
data_lines = []
if has_begin and has_end:
    in_block = False
    for line in raw_lines:
        s = line.strip()
        if s.startswith("<begin_xyz>"):
            in_block = True
            continue
        if s.startswith("<end_xyz>"):
            in_block = False
            continue
        if in_block and s and not s.startswith("<"):
            data_lines.append(s)
else:
    # No markers: take every non-empty, non-tagged line
    for line in raw_lines:
        s = line.strip()
        if s and not s.startswith("<"):
            data_lines.append(s)

# 5. Load into a DataFrame
txt = "\n".join(data_lines)
data = pd.read_csv(
    StringIO(txt),
    header=None,
    names=["longitude", "latitude", "depth"],
    sep=r"\s+",
)
data_list = data.to_dict(orient="records")

# 6. Sample points if too many
if len(data_list) > MAX_POINTS:
    data_sample = random.sample(data_list, MAX_POINTS)
else:
    data_sample = data_list

# 7. Compute map center
center_lat = np.mean([pt["latitude"]  for pt in data_sample])
center_lon = np.mean([pt["longitude"] for pt in data_sample])

# 8. Initialize the map
grid_map = Map(center=(center_lat, center_lon), zoom=10)

# 9. Prepare colormap
cmap = cm.viridis
norm = Normalize(
    vmin=min(pt["depth"] for pt in data_sample),
    vmax=max(pt["depth"] for pt in data_sample),
)

def value_to_color(val):
    return to_hex(cmap(norm(val)))

# 10. Plot circles
for pt in data_sample:
    circle = Circle(
        location=(pt["latitude"], pt["longitude"]),
        radius=30,
        color=value_to_color(pt["depth"]),
        fill_color=value_to_color(pt["depth"]),
        fill_opacity=0.7,
        weight=1,
    )
    grid_map.add_layer(circle)

# 11. Display
grid_map

# 3. Grid
Load a previously generated Mohid grid file or create a new grid 

## 3.1 Load a previously generated Mohid grid file

In [ ]:
# Load grid from file
file_path = Path(r"mohid_grid.grd") #define the path to your grid

# Load a MOHID grid data file
with open(file_path, 'r') as f:
    lines = f.readlines()

grid_data = []
x_coords = []
y_coords = []
polylines = []
n_rows, n_cols = None, None  # Use None to detect missing values
start_reading_xx = False
start_reading_yy = False
start_reading_grid = False

# 2. Initialize and display the map
#grid_map = Map(center=(0.0, 0.0), zoom=2)
#display(grid_map)

for line in lines:
    line = line.strip()  # Remove leading and trailing spaces
    parts = line.split()

    if line.startswith("ILB_IUB"):
        n_rows = int(parts[3]) 
    elif line.startswith("JLB_JUB"):
        n_cols = int(parts[3])
    elif line.startswith("ORIGIN"):
        x0 = float(parts[2])
        y0 = float(parts[3])
    elif line.startswith("DX"):
        dx = float(parts[2])
    elif line.startswith("DY"):
        dy = float(parts[2])
    elif line.startswith("GRID_ANGLE"):
        grid_angle = float(parts[2])
    elif "<BeginXX>" in line:
        start_reading_xx = True
        continue
    elif "<EndXX>" in line:
        start_reading_xx = False
    elif start_reading_xx:
        try:
            x_coords.append(float(line))
        except ValueError:
            print(f"Warning: Skipping invalid line -> {line}")
    elif "<BeginYY>" in line:
        start_reading_yy = True
        continue
    elif "<EndYY>" in line:
        start_reading_yy = False
    elif start_reading_yy:
        try:
            y_coords.append(float(line))
        except ValueError:
            print(f"Warning: Skipping invalid line -> {line}")

# Debugging prints
print(f"Extracted Dimensions: n_rows={n_rows}, n_cols={n_cols}")

grid_map = Map(center=(y0, x0), zoom=8)
display(grid_map)

# 5. Grid generation function
def generate_grid(x0, y0):
    global polylines, Xr, Yr
    
    # Create a regular mesh
    x_coords = np.linspace(x0, x0 + dx * n_cols, n_cols + 1)
    y_coords = np.linspace(y0, y0 + dy * n_rows, n_rows + 1)
    X, Y = np.meshgrid(x_coords, y_coords)
    
    # Precompute rotation
    theta = radians(grid_angle)
    ct, st = cos(theta), sin(theta)
    
    # Rotate each mesh point about (x0, y0)
    Xr = np.empty_like(X)
    Yr = np.empty_like(Y)
    for j in range(n_rows + 1):
        for i in range(n_cols + 1):
            dx_i = X[j, i] - x0
            dy_j = Y[j, i] - y0
            Xr[j, i] = x0 + (dx_i * ct - dy_j * st)
            Yr[j, i] = y0 + (dx_i * st + dy_j * ct)
    
    # Remove existing grid layers
    for layer in polylines:
        if layer in grid_map.layers:
            grid_map.remove_layer(layer)
    polylines.clear()
    
    # Build and draw rotated grid lines
    grid_group = LayerGroup()
    # Horizontal lines
    for j in range(n_rows + 1):
        pts = [(Yr[j, i], Xr[j, i]) for i in range(n_cols + 1)]
        grid_group.add_layer(Polyline(locations=pts, color="blue", weight=1))
    # Vertical lines
    for i in range(n_cols + 1):
        pts = [(Yr[j, i], Xr[j, i]) for j in range(n_rows + 1)]
        grid_group.add_layer(Polyline(locations=pts, color="blue", weight=1))
    
    grid_map.add_layer(grid_group)
    polylines.append(grid_group)
    
if not x_coords:
    #x_coords = np.linspace(x0, x0 + dx * n_cols, n_cols+1)
    #y_coords = np.linspace(y0, y0 + dy * n_rows, n_rows+1)
    generate_grid(x0, y0)
else:
    x_coords = np.array(x_coords) + x0
    y_coords = np.array(y_coords) + y0

    Xr, Yr = np.meshgrid(x_coords, y_coords)
    
# Ensure grid dimensions are valid
if n_rows is None or n_cols is None:
    raise ValueError("Grid dimensions could not be determined from the file.")
    
print(f"Loaded grid")

## 3.2 Create a new grid

### 3.2.1 Get grid dimensions and spacing

In [ ]:
nx = int(input("Enter the number of cells in the x-direction (nx): "))
ny = int(input("Enter the number of cells in the y-direction (ny): "))

In [ ]:
dx = float(input("Enter the cell size in the x-direction (dx, in degrees): "))
dy = float(input("Enter the cell size in the y-direction (dy, in degrees): "))

### 3.2.2 Grid generation
Click on the map to create the grid

In [ ]:
# Full interactive rotatable grid with manual angle input
import numpy as np
from math import radians, cos, sin
from ipyleaflet import Map, Marker, Polyline, LayerGroup
from ipywidgets import HTML, FloatText

# 1. Parameters and state holders
marker = None
polylines = []
x0 = None
y0 = None
grid_angle = 0.0     # rotation angle in degrees

# Initialize the interactive map
try:
    grid_map
except NameError:
    grid_map = Map(center=(0.0, 0.0), zoom=2)

# Display the map
display(grid_map)

# 3. Instructions widget
instructions = HTML("""
<h4>Interactive Rotatable Grid</h4>
<ol>
  <li>Click on the map to set the grid origin.</li>
  <li>Enter a specific angle (in degrees) to rotate the grid.</li>
</ol>
""")
display(instructions)

# 4. Angle input widget
angle_input = FloatText(
    value=0.0,
    description='Angle (°):',
    step=0.1
)
display(angle_input)

def on_angle_change(change):
    global grid_angle
    grid_angle = change['new']
    if x0 is not None and y0 is not None:
        generate_grid(x0, y0)

angle_input.observe(on_angle_change, names='value')

# 5. Grid generation function
def generate_grid(x0, y0):
    global polylines, Xr, Yr
    
    # Create a regular mesh
    x_coords = np.linspace(x0, x0 + dx * nx, nx + 1)
    y_coords = np.linspace(y0, y0 + dy * ny, ny + 1)
    X, Y = np.meshgrid(x_coords, y_coords)
    
    # Precompute rotation
    theta = radians(grid_angle)
    ct, st = cos(theta), sin(theta)
    
    # Rotate each mesh point about (x0, y0)
    Xr = np.empty_like(X)
    Yr = np.empty_like(Y)
    for j in range(ny + 1):
        for i in range(nx + 1):
            dx_i = X[j, i] - x0
            dy_j = Y[j, i] - y0
            Xr[j, i] = x0 + (dx_i * ct - dy_j * st)
            Yr[j, i] = y0 + (dx_i * st + dy_j * ct)
    
    # Remove existing grid layers
    for layer in polylines:
        if layer in grid_map.layers:
            grid_map.remove_layer(layer)
    polylines.clear()
    
    # Build and draw rotated grid lines
    grid_group = LayerGroup()
    # Horizontal lines
    for j in range(ny + 1):
        pts = [(Yr[j, i], Xr[j, i]) for i in range(nx + 1)]
        grid_group.add_layer(Polyline(locations=pts, color="blue", weight=1))
    # Vertical lines
    for i in range(nx + 1):
        pts = [(Yr[j, i], Xr[j, i]) for j in range(ny + 1)]
        grid_group.add_layer(Polyline(locations=pts, color="blue", weight=1))
    
    grid_map.add_layer(grid_group)
    polylines.append(grid_group)

# 6. Click handler to set origin and draw grid
def handle_map_click(**kwargs):
    global marker, x0, y0
    if kwargs.get("type") == "click":
        lat, lng = kwargs["coordinates"]
        x0, y0 = lng, lat
        
        # Remove old marker
        if marker is not None and marker in grid_map.layers:
            grid_map.remove_layer(marker)
        
        # Add new marker
        marker = Marker(location=(y0, x0))
        grid_map.add_layer(marker)
        
        # Generate and display the grid
        generate_grid(x0, y0)

grid_map.on_interaction(handle_map_click)

### 3.2.3 Save the grid to a MOHID-compatible file

In [ ]:
# Get current date and time
now = datetime.now()

# Format the date and time
formatted_date_time = now.strftime("%d-%m-%Y %H:%M:%S")

output_file = "mohid_grid.grd"
with open(output_file, "w") as f:
    f.write("PROJ4_STRING              : +proj=longlat +datum=WGS84 +no_defs\n")
    f.write("COMENT1                   : Grid generated by MOHID Jupyter Notebook\n")
    f.write("COMENT1                   : Generation Time: " + formatted_date_time + "\n")
    f.write("LATITUDE                  : " + str(y0) + "\n")
    f.write("LONGITUDE                 : " + str(x0) + "\n")
    f.write("COORD_TIP                 : 4\n")
    f.write("ILB_IUB                   : 1 " + str(ny) + "\n")
    f.write("JLB_JUB                   : 1 " + str(nx) + "\n")
    f.write("ORIGIN                    : " + str(x0) + " " + str(y0) + "\n")
    f.write("GRID_ANGLE                : " + str(grid_angle) + "\n")
    f.write("CONSTANT_SPACING_X        : 1\n")
    f.write("CONSTANT_SPACING_Y        : 1\n")
    f.write("DX                        : " + str(dx) + "\n")
    f.write("DY                        : " + str(dy) + "\n")

print(f"\nGrid saved to {output_file}")

# 4. Coastline

## 4.1 Load coastline
Load your coastline shapefile or download it from:
 - GSHHG (coarse resolution)
 - OSM (fine resolution)

### 4.1.1 Load your coastline
Shapefile or MOHID polygon file

#### Shapefile

In [ ]:
coastline_shapefile = "Coastlines/Para_3_completo.shp" #define the path to your coastline data

#### MOHID polygon file 

In [ ]:
# Paths
coastline_file = Path(r"")  #define the path to your coastline data

out_dir = Path("Coastlines")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Read and parse coordinates
coords = []
with coastline_file.open() as f:
    for line in f:
        line = line.strip()
        # Skip empty lines and polygon markers
        if not line or line.startswith("<beginpolygon>") or line.startswith("<endpolygon>"):
            continue
        parts = line.split()
        if len(parts) != 3:
            continue
        lon, lat, _ = parts
        coords.append((float(lon), float(lat)))

# 2. Ensure polygon closure
if coords and coords[0] != coords[-1]:
    coords.append(coords[0])

# 3. Build the Shapely polygon and wrap in a GeoDataFrame
polygon = Polygon(coords)
gdf = gpd.GeoDataFrame({"geometry": [polygon]}, crs="EPSG:4326")

# 4. Export to Shapefile (and GeoJSON as an example)
coastline_shapefile = out_dir / "output_polygon.shp"
gdf.to_file(coastline_shapefile, driver="ESRI Shapefile")

print(f"Shapefile written to: {coastline_shapefile.resolve()}")

### 4.1.2 GSHHG Coastline Data
Download (first time) or load GSHHG Coastline Data

In [ ]:
# Define URL and local paths
gshhg_url = "https://www.soest.hawaii.edu/pwessel/gshhg/gshhg-shp-2.3.7.zip"
zip_path = "gshhg_shapefiles.zip"
extract_path = "gshhg_data"

# Download the file if not exists
if not os.path.exists(zip_path):
    print("Downloading GSHHG data...")
    response = requests.get(gshhg_url)
    with open(zip_path, "wb") as f:
        f.write(response.content)

# Extract the files
if not os.path.exists(extract_path):
    print("Extracting GSHHG data...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

print("GSHHG data is ready.")

coastline_shapefile = os.path.join(extract_path, "GSHHS_shp", "f", "GSHHS_f_L1.shp") #if using downloaded GSHHG coastline data


### 4.1.3 OSM Coastline Data
Download (first time) or load OSM Coastline Data

In [ ]:
import os
from zipfile import ZipFile
import requests
import numpy as np
import geopandas as gpd
from shapely.geometry import box
from ipyleaflet import Map, GeoJSON

# 1. Download OSM land-polygons (WGS84, split into tiles)
url      = "https://osmdata.openstreetmap.de/download/land-polygons-split-4326.zip"
zipfile  = "land-polygons.zip"
out_dir  = "land-polygons"

if not os.path.exists(zipfile):
    print("Downloading land polygons…")
    r = requests.get(url, stream=True)
    with open(zipfile, "wb") as f:
        for chunk in r.iter_content(1024):
            f.write(chunk)

if not os.path.exists(out_dir):
    print("Extracting…")
    with ZipFile(zipfile, "r") as z:
        z.extractall(out_dir)

# 2. Read the big land-polygons layer
#    The shapefile is called 'land_polygons.shp' inside the unzip folder
coastline_shapefile = os.path.join(out_dir, "land-polygons-split-4326", "land_polygons.shp")


## 4.2 Filter Coastlines for Grid Region

In [ ]:
coastlines = gpd.read_file(coastline_shapefile)

# Define a bounding box
np_x = np.array(Xr)
np_y = np.array(Yr)

c = 3
x_min=np.min(np_x)-dx*c; y_min=np.min(np_y)-dy*c; x_max=np.max(np_x)+dx*c; y_max=np.max(np_y)+dy*c

bbox = [x_min, y_min, x_max, y_max] #[min_lon, min_lat, max_lon, max_lat]

# Create a bounding box geometry
bbox_geom = gpd.GeoDataFrame(geometry=[box(*bbox)], crs=coastlines.crs)

# Filter coastlines for the bounding box
filtered_coastlines = gpd.overlay(coastlines, bbox_geom, how='intersection')

## 4.3 Show coastline

In [ ]:
# Create an interactive map centered in the region
center_lat = (bbox[1] + bbox[3]) / 2
center_lon = (bbox[0] + bbox[2]) / 2
m = Map(center=(center_lat, center_lon), zoom=8)

# Add coastlines
geo_layer = GeoData(geo_dataframe=filtered_coastlines, style={"color": "blue", "weight": 1})

# Add to map
m.add_layer(geo_layer)

# Display the map
m

# 5. Bathymetry
Load a previously generated Mohid griddata file or create a new griddata

## 5.1 Load a previously generated Mohid griddata file

In [ ]:
# Load grid data from file
file_path = "mohid_griddata_from_shapefile.dat" #define the path for your griddata file

# Load a MOHID grid data file
with open(file_path, 'r') as f:
    lines = f.readlines()

grid_data = []
x_coords = []
y_coords = []
nx, ny = None, None  # Use None to detect missing values
start_reading_xx = False
start_reading_yy = False
start_reading_grid = False

for line in lines:
    line = line.strip()  # Remove leading and trailing spaces
    parts = line.split()

    if line.startswith("ILB_IUB"):
        ny = int(parts[3]) 
    elif line.startswith("JLB_JUB"):
        nx = int(parts[3])
    elif line.startswith("ORIGIN"):
        x0 = float(parts[2])
        y0 = float(parts[3])
    elif line.startswith("DX"):
        dx = float(parts[2])
    elif line.startswith("DY"):
        dy = float(parts[2])
    elif line.startswith("GRID_ANGLE"):
        grid_angle = float(parts[2])        
    elif "<BeginXX>" in line:
        start_reading_xx = True
        continue
    elif "<EndXX>" in line:
        start_reading_xx = False
    elif start_reading_xx:
        try:
            x_coords.append(float(line))
        except ValueError:
            print(f"Warning: Skipping invalid line -> {line}")
    elif "<BeginYY>" in line:
        start_reading_yy = True
        continue
    elif "<EndYY>" in line:
        start_reading_yy = False
    elif start_reading_yy:
        try:
            y_coords.append(float(line))
        except ValueError:
            print(f"Warning: Skipping invalid line -> {line}")
    elif "<BeginGridData2D>" in line:
        start_reading_grid = True
        continue
    elif "<EndGridData2D>" in line:
        start_reading_grid = False
    elif start_reading_grid:
        try:
            grid_data.append(float(line))
        except ValueError:
            print(f"Warning: Skipping invalid line -> {line}")

# Debugging prints
print(f"Extracted Dimensions: n_rows={ny}, n_cols={nx}")
print(f"Grid Data Length: {len(grid_data)}")

# Ensure grid dimensions are valid
if ny is None or nx is None:
    raise ValueError("Grid dimensions could not be determined from the file.")
    
# Precompute rotation matrix
theta = radians(grid_angle)
R = np.array([[cos(theta), -sin(theta)],
              [sin(theta),  cos(theta)]])

if not x_coords:
    # Case 1: Regular mesh
    x_coords = np.linspace(x0, x0 + dx * nx, nx + 1)
    y_coords = np.linspace(y0, y0 + dy * ny, ny + 1)
else:
    # Case 2: Provided coordinates
    x_coords = np.array(x_coords) + x0
    y_coords = np.array(y_coords) + y0

# Build mesh
X, Y = np.meshgrid(x_coords, y_coords)

# Flatten for vectorized rotation
points = np.vstack([X.ravel() - x0, Y.ravel() - y0])

# Apply rotation
rotated = R @ points

# Reshape back to grid
Xr = rotated[0].reshape(X.shape) + x0
Yr = rotated[1].reshape(Y.shape) + y0

# Check if data size matches expected shape
expected_size = ny * nx
if len(grid_data) != expected_size:
    raise ValueError(f"Mismatch: Grid data size {len(grid_data)} does not match expected ({expected_size}).")

# Convert to NumPy array and reshape correctly
zi = np.array(grid_data).reshape(ny, nx)
    
print(f"Loaded grid data shape: {zi.shape}")

## 5.2 Create a new griddata
Interpolate bathymetry from the xyz data (section 2 or the GEBCO_2026 download in 5.2.1) and the coastline (section 4)

### 5.2.1 Download GEBCO_2026 bathymetry for the grid area (optional)
Downloads the [GEBCO_2026 Grid](https://www.gebco.net/data-products/gridded-bathymetry-data) (15 arc-second, ~460 m) for the area covered by the grid (`Xr`, `Yr` from section 3) and stores it in `data` (longitude, latitude, depth positive downward), replacing section 2.

- First it tries **OPeNDAP** on the CEDA archive, which transfers only the grid area.
- If OPeNDAP is unavailable, it downloads the **global netCDF zip (~4 GB, 7 GB unzipped)** once into `GEBCO_2026/` and cuts the area locally. Later runs reuse the cached file.
- Requires `xarray` and `netCDF4` (`conda install -c conda-forge xarray netcdf4`).

Citation: *GEBCO Compilation Group (2026) GEBCO 2026 Grid.* See the [terms of use](https://www.gebco.net/data-products/gridded-bathymetry/terms-of-use).

In [ ]:
# ───────── USER CONFIG ─────────
gebco_variant    = "ice_surface_elevation"  # or "sub_ice_topography_bathymetry"
gebco_dir        = Path("GEBCO_2026")       # download / output folder
buffer_cells     = 3       # margin around the grid, in grid cells (same idea as 4.2)
drop_land_points = True    # drop GEBCO points with elevation >= 0 (land) before interpolation
save_xyz         = True    # also save a MOHID .xyz file (can be reloaded in section 2)
keep_global_file = True    # keep the global 7 GB netCDF for later runs (fallback route only)

try:
    import xarray as xr
except ImportError as err:
    raise ImportError("This cell needs xarray and netCDF4:\n"
                      "    conda install -c conda-forge xarray netcdf4") from err
import zipfile as _zipfile   # alias: cell 4.1.3 reassigns the name `zipfile`

_gebco_base    = "https://dap.ceda.ac.uk/bodc/gebco/global/gebco_2026"                 # file downloads
_gebco_dap_base = "https://dap.ceda.ac.uk/thredds/dodsC/bodc/gebco/global/gebco_2026"  # OPeNDAP (THREDDS)
_gebco_files = {
    "ice_surface_elevation":         ("GEBCO_2026.nc",         "GEBCO_2026.zip"),
    "sub_ice_topography_bathymetry": ("GEBCO_2026_sub_ice.nc", "GEBCO_2026_sub_ice.zip"),
}
_nc_name, _zip_name = _gebco_files[gebco_variant]
gebco_opendap_url = f"{_gebco_dap_base}/{gebco_variant}/netcdf/{_nc_name}"
gebco_zip_url     = f"{_gebco_base}/{gebco_variant}/netcdf/{_zip_name}?download=1"
gebco_dir.mkdir(parents=True, exist_ok=True)

# 1. Area covered by the grid (works for rotated grids) + margin
_Xg, _Yg = np.asarray(Xr, dtype=float), np.asarray(Yr, dtype=float)
_cell = max(np.nanmax(np.hypot(np.diff(_Xg, axis=1), np.diff(_Yg, axis=1))),
            np.nanmax(np.hypot(np.diff(_Xg, axis=0), np.diff(_Yg, axis=0))))
_margin = max(buffer_cells * _cell, 2 / 240)       # at least 2 GEBCO cells
_to180 = lambda v: ((v + 180.0) % 360.0) - 180.0   # GEBCO longitudes are -180..180
g_lon_min, g_lon_max = _to180(np.nanmin(_Xg) - _margin), _to180(np.nanmax(_Xg) + _margin)
g_lat_min = max(np.nanmin(_Yg) - _margin, -90.0)
g_lat_max = min(np.nanmax(_Yg) + _margin,  90.0)
if g_lon_min > g_lon_max:
    raise ValueError("The grid area crosses the 180° meridian; this cell does not handle that case.")
print(f"GEBCO area: lon {g_lon_min:.4f} to {g_lon_max:.4f}, lat {g_lat_min:.4f} to {g_lat_max:.4f}")


def _gebco_subset(ds):
    sub = ds[["elevation"]].sel(lon=slice(g_lon_min, g_lon_max), lat=slice(g_lat_min, g_lat_max))
    if sub.sizes["lon"] == 0 or sub.sizes["lat"] == 0:
        raise ValueError("Empty GEBCO subset - check the grid coordinates.")
    return sub.load()


def _gebco_download(url, dest, chunk=8 * 1024 * 1024):
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = None
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        bar = tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) if tqdm else None
        done, last = 0, 0
        with open(tmp, "wb") as f:
            for block in r.iter_content(chunk_size=chunk):
                f.write(block)
                done += len(block)
                if bar:
                    bar.update(len(block))
                elif done - last > 200 * 1024**2:
                    print(f"  {done / 1024**3:.2f} GB downloaded", end="\r")
                    last = done
        if bar:
            bar.close()
    tmp.replace(dest)


# 2. Get the subset: local global file if already downloaded, else OPeNDAP, else download
_nc_path = gebco_dir / _nc_name
gebco = None
if _nc_path.exists():
    print(f"Using local global file: {_nc_path}")
    with xr.open_dataset(_nc_path) as _ds:
        gebco = _gebco_subset(_ds)
else:
    try:
        print(f"Trying OPeNDAP: {gebco_opendap_url}")
        with xr.open_dataset(gebco_opendap_url, engine="netcdf4") as _ds:
            gebco = _gebco_subset(_ds)
        print("OPeNDAP subset OK.")
    except Exception as err:
        print(f"OPeNDAP unavailable ({type(err).__name__}: {err})\nFalling back to the global file.")
if gebco is None:
    if not _nc_path.exists():
        _zip_path = gebco_dir / _zip_name
        if not _zip_path.exists():
            print(f"Downloading global GEBCO_2026 zip (~4 GB): {gebco_zip_url}")
            _gebco_download(gebco_zip_url, _zip_path)
        print("Extracting netCDF from zip ...")
        with _zipfile.ZipFile(_zip_path) as z:
            member = next(m for m in z.namelist() if m.lower().endswith(".nc"))
            z.extract(member, gebco_dir)
            (gebco_dir / member).replace(_nc_path)
        _zip_path.unlink()
    with xr.open_dataset(_nc_path) as _ds:
        gebco = _gebco_subset(_ds)
    if not keep_global_file:
        _nc_path.unlink()

# 3. Save the subset as netCDF
_tag = f"lon{g_lon_min:.3f}_{g_lon_max:.3f}_lat{g_lat_min:.3f}_{g_lat_max:.3f}"
gebco.attrs.update({"source": "GEBCO_2026 Grid, GEBCO Compilation Group (2026)",
                    "url": "https://www.gebco.net/data-products/gridded-bathymetry-data"})
gebco_nc_out = gebco_dir / f"GEBCO_2026_{_tag}.nc"
gebco.to_netcdf(gebco_nc_out, encoding={"elevation": {"zlib": True, "complevel": 4}})
print(f"Saved netCDF: {gebco_nc_out}  ({gebco.sizes['lat']} x {gebco.sizes['lon']} cells)")

# 4. Build `data` (same columns as section 2). MOHID depth is positive downward.
_lon2d, _lat2d = np.meshgrid(gebco["lon"].values, gebco["lat"].values)
_depth = -gebco["elevation"].values.astype(float)
_keep = np.isfinite(_depth)
if save_xyz:
    gebco_xyz_out = gebco_dir / f"GEBCO_2026_{_tag}.xyz"
    with open(gebco_xyz_out, "w") as f:
        f.write("<begin_xyz>\n")
        np.savetxt(f, np.column_stack([_lon2d[_keep], _lat2d[_keep], _depth[_keep]]),
                   fmt="%.6f %.6f %.2f")
        f.write("<end_xyz>\n")
    print(f"Saved XYZ   : {gebco_xyz_out}")
if drop_land_points:
    _keep &= _depth > 0
data = pd.DataFrame({"longitude": _lon2d[_keep], "latitude": _lat2d[_keep], "depth": _depth[_keep]})
data.attrs["source"] = "GEBCO_2026"   # lets 5.2.2 use the GEBCO grid directly ("area_mean")
print(f"`data` ready for interpolation: {len(data):,} points "
      f"(depth {data['depth'].min():.1f} to {data['depth'].max():.1f} m)")

# 5. Quick look: GEBCO depth with the grid outline
fig, ax = plt.subplots(figsize=(8, 7))
(-gebco["elevation"]).where(gebco["elevation"] < 0).plot(
    ax=ax, cmap="viridis_r", cbar_kwargs={"label": "Depth (m)"})
_ox = np.concatenate([_Xg[0, :], _Xg[:, -1], _Xg[-1, ::-1], _Xg[::-1, 0]])
_oy = np.concatenate([_Yg[0, :], _Yg[:, -1], _Yg[-1, ::-1], _Yg[::-1, 0]])
ax.plot(_ox, _oy, "r-", lw=1.5, label="Model grid")
ax.set_title("GEBCO_2026 bathymetry"); ax.set_aspect("equal"); ax.legend(loc="lower right")
plt.show()

### 5.2.2 Interpolate bathymetry onto the grid
Uses `data` from section 2 or 5.2.1, plus the coastline from section 4.

`interp_method`:
- `"auto"` (default): `"area_mean"` when `data` comes from the GEBCO download (5.2.1). For scattered xyz data (section 2), `"cell_mean"` when there are at least `min_pts_per_cell` points per water cell, otherwise `"nearest"`.
- `"area_mean"` (GEBCO only): area-weighted mean depth over each cell. Each cell is sampled with `n_sub` × `n_sub` sub-points and GEBCO is interpolated bilinearly at each one, so the result doesn't depend on how the grid lines up with GEBCO, and rotated grids work too.
- `"cell_mean"`: average of all points inside each cell. Water cells with no points are filled with the nearest value.
- `"nearest"`: value of the closest point. Suited to sparse or irregular survey data.
- `"linear"`: linear interpolation. Use this when the grid is finer than the data.

Then, in this order:
1. **Gaussian smoothing** (`sigma`). Automatic: off for the averaging methods, 1 cell for `nearest`/`linear`.
2. **Spike and pit removal** (`despike`). A water cell is replaced by the median of its water neighbours only if it is deeper than all of them (pit) or shallower than all of them (spike) by more than `max(spike_abs, spike_rel × median)`. Features two or more cells wide, such as channels, ridges and banks, are kept.
3. **Minimum depth** (`min_depth`).
4. **Slope limiting** (`rx0_max`, typically 0.2). Where rx0 = |h₁ − h₂| / (h₁ + h₂) between neighbouring cells exceeds the limit, the two depths are moved toward each other, keeping their sum (and so the water volume) unchanged. It is applied only where the shallower cell is shallower than `sigma_layer_depth`: with sigma layers in the top 20 m and Cartesian layers below, that is the only place where sigma layers follow the seabed. Set `sigma_layer_depth = None` for a full-sigma grid, or `rx0_max = None` to switch slope limiting off.
5. **Removal of small isolated water patches** (`min_patch`).

**Optional: merge local chart or survey data** (`chart_xyz`). Give an xyz file (lon lat depth, depth positive down). Its soundings are averaged per cell, short gaps between them are filled by linear interpolation (`chart_max_gap` cells), Where soundings are dense (at least `chart_min_density`, by default 40%, of the water cells within `chart_max_gap` cells have soundings), the chart data replace the base depths, and a transition `chart_blend` cells wide blends back to GEBCO around them. Isolated soundings only adjust their own cell, with a weight proportional to the local density, so a single sounding can't override GEBCO over a wide area. Chart depths are referred to chart datum, while GEBCO is roughly mean sea level, so set `chart_datum_below_msl` (Z0) to the chart's datum level. For DHN charts this is the *nível de redução*, given per area by the tide station data. The merge runs before despiking, minimum depth and slope limiting.

**Optional: estuary mask** (`estuary_polygons`). A shapefile/GeoJSON path, or a list of polygons `[[(lon, lat), ...], ...]`:
- `estuary_mode = "land"` removes those areas from the grid (-99), e.g. when the model is for the shelf only.
- `estuary_mode = "no_rx0"` keeps them but doesn't apply slope limiting inside them, e.g. when they contain chart data you don't want reshaped.

In [ ]:
import warnings
# ───────── USER CONFIG ─────────
interp_method    = "auto"  # "auto", "area_mean", "cell_mean", "nearest" or "linear" (see the description above)
min_pts_per_cell = 2       # "auto" (scattered data): "cell_mean" when there are at least this many points per water cell
n_sub            = None    # "area_mean": sub-points per cell side (None = automatic from grid / GEBCO spacing)
sigma            = None    # Gaussian smoothing, in cells. None = automatic: 0 for area_mean/cell_mean, 1.0 otherwise
min_depth        = 1.0     # water cells get at least this depth (m); None = no limit
despike          = True    # replace isolated single-cell spikes/pits by the median of their water neighbours
spike_rel        = 0.5     # ...when they stand out from ALL neighbours by more than this fraction of the median
spike_abs        = 2.0     # ...and by more than this many metres
rx0_max          = 0.3     # max slope factor |h1-h2|/(h1+h2) between neighbours; None = no slope limiting
sigma_layer_depth = 20.0   # thickness (m) of the sigma domain; rx0 is limited only where the shallower cell is < this
                           # (None = limit rx0 everywhere, e.g. for a full-sigma vertical grid)
min_patch        = 20      # min cells to keep a water patch
nodata_val       = -99.0

# ───────── LOCAL CHART / SURVEY DATA (optional) ─────────
#chart_xyz        = None    # path to an xyz file (lon lat depth, depth positive down; <begin_xyz> tags optional)
chart_xyz        = Path("Data/chart_combined_v4.xyz")    # path to an xyz file (lon lat depth, depth positive down; <begin_xyz> tags optional)

chart_datum_below_msl = 0.8  # Z0 (m): how far chart datum lies below mean sea level; added to chart depths
                           # (GEBCO is ~MSL; DHN charts use "nível de redução", so Z0 is usually > 0)
chart_max_gap    = 2       # fill cells without soundings by interpolating chart data up to this many cells away
chart_blend      = 3       # width (cells) of the transition from chart data to GEBCO / the base data
chart_min_density = 0.4   # chart data fully replace the base data only where at least this fraction of the water cells
                           # within chart_max_gap cells have soundings; isolated soundings get a proportionally lower weight

# ───────── ESTUARY MASK (optional) ─────────
estuary_polygons = None    # shapefile/GeoJSON path, or a list of polygons [[(lon, lat), ...], ...]
estuary_mode     = "no_rx0"  # "land": remove estuaries from the grid (-99)
                           # "no_rx0": keep them, but don't apply slope limiting inside them

# Initialize the grid with -99
zi = np.full((Xr.shape[0] - 1, Xr.shape[1] - 1), nodata_val, dtype=float)
n_rows_c, n_cols_c = zi.shape

# Compute cell-centers (average of four rotated corners)
xc = 0.25*(Xr[:-1, :-1] + Xr[:-1, 1:] + Xr[1:, :-1] + Xr[1:, 1:])
yc = 0.25*(Yr[:-1, :-1] + Yr[:-1, 1:] + Yr[1:, :-1] + Yr[1:, 1:])

# Water cells = cell centres outside the coastline polygons
water_mask = ~contains_xy(filtered_coastlines.union_all(), xc, yc)

# Estuary mask (optional)
estuary_mask = np.zeros_like(water_mask)
if estuary_polygons is not None:
    if isinstance(estuary_polygons, (str, Path)):
        _est = gpd.read_file(estuary_polygons).to_crs("EPSG:4326").union_all()
    else:
        from shapely.geometry import Polygon as _SPolygon   # (ipyleaflet also has a Polygon)
        _est = unary_union([_SPolygon(p) for p in estuary_polygons])
    estuary_mask = water_mask & contains_xy(_est, xc, yc)
    if estuary_mode == "land":
        water_mask = water_mask & ~estuary_mask
    elif estuary_mode != "no_rx0":
        raise ValueError(f"Unknown estuary_mode: {estuary_mode!r}")
    print(f"Estuary mask: {estuary_mask.sum():,} water cells ({estuary_mode})")

# Data points (from section 2 or 5.2.1), NaNs removed
mask = ~np.isnan(data['longitude']) & ~np.isnan(data['latitude']) & ~np.isnan(data['depth'])
lons   = data['longitude'][mask].to_numpy(dtype=float)
lats   = data['latitude'][mask].to_numpy(dtype=float)
depths = data['depth'][mask].to_numpy(dtype=float)

def _nearest(x, y):
    return griddata((lons, lats), depths, (x, y), method="nearest")

# ---------------------------------------------------------------- choose method
data_is_gebco = data.attrs.get("source") == "GEBCO_2026" and "gebco" in globals()
method = interp_method
if method == "area_mean" and not data_is_gebco:
    raise ValueError('"area_mean" needs the GEBCO grid: run 5.2.1 first (section 2 data are scattered points).')

if method in ("auto", "cell_mean"):
    # Locate each data point in the grid (row i, column j); works for rotated grids
    _ang = np.arctan2(Yr[0, -1] - Yr[0, 0], Xr[0, -1] - Xr[0, 0])
    _ca, _sa = np.cos(_ang), np.sin(_ang)
    def _unrotate(x, y):
        ddx, ddy = np.asarray(x) - Xr[0, 0], np.asarray(y) - Yr[0, 0]
        return ddx * _ca + ddy * _sa, -ddx * _sa + ddy * _ca
    _u_edges, _ = _unrotate(Xr[0, :], Yr[0, :])
    _, _v_edges = _unrotate(Xr[:, 0], Yr[:, 0])
    _su = np.sign(_u_edges[-1] - _u_edges[0]) or 1.0
    _sv = np.sign(_v_edges[-1] - _v_edges[0]) or 1.0
    _pu, _pv = _unrotate(lons, lats)
    pt_j = np.searchsorted(_su * _u_edges, _su * _pu, side="right") - 1
    pt_i = np.searchsorted(_sv * _v_edges, _sv * _pv, side="right") - 1
    inside = (pt_i >= 0) & (pt_i < n_rows_c) & (pt_j >= 0) & (pt_j < n_cols_c)

if method == "auto":
    if data_is_gebco:
        method = "area_mean"
    else:
        _in_water = inside.copy()
        _in_water[inside] = water_mask[pt_i[inside], pt_j[inside]]
        method = "cell_mean" if _in_water.sum() >= min_pts_per_cell * max(water_mask.sum(), 1) else "nearest"
if sigma is None:
    sigma = 0.0 if method in ("area_mean", "cell_mean") else 1.0
print(f"Interpolation method: {method}  ({zi.size:,} cells, {water_mask.sum():,} water cells), "
      f"Gaussian sigma = {sigma}")

# ---------------------------------------------------------------- interpolate
if method == "area_mean":
    # Area-weighted mean of the GEBCO grid over each cell: n_sub x n_sub sub-points per cell,
    # placed by bilinear mapping of the 4 (rotated) corners, GEBCO bilinear-interpolated at each.
    from scipy.interpolate import RegularGridInterpolator
    _glon = gebco["lon"].values
    _glat = gebco["lat"].values
    _gdep = -gebco["elevation"].values.astype(float)        # depth positive downward, land < 0
    _rgi = RegularGridInterpolator((_glat, _glon), _gdep, method="linear",
                                   bounds_error=False, fill_value=np.nan)
    if n_sub is None:
        _cell_deg = max(np.hypot(Xr[0, 1] - Xr[0, 0], Yr[0, 1] - Yr[0, 0]),
                        np.hypot(Xr[1, 0] - Xr[0, 0], Yr[1, 0] - Yr[0, 0]))
        _gstep = abs(_glon[1] - _glon[0])
        n_sub = int(np.clip(np.ceil(2 * _cell_deg / _gstep), 2, 12))
    _f = (np.arange(n_sub) + 0.5) / n_sub
    _s, _t = [a.ravel() for a in np.meshgrid(_f, _f)]       # fractional positions inside a cell
    _w00, _w10, _w01, _w11 = (1-_s)*(1-_t), _s*(1-_t), (1-_s)*_t, _s*_t
    _wi, _wj = np.nonzero(water_mask)
    _area = np.full(_wi.size, np.nan)
    _chunk = max(1, 2_000_000 // (n_sub * n_sub))
    for k in range(0, _wi.size, _chunk):
        ii, jj = _wi[k:k+_chunk, None], _wj[k:k+_chunk, None]
        px = _w00*Xr[ii, jj] + _w10*Xr[ii, jj+1] + _w01*Xr[ii+1, jj] + _w11*Xr[ii+1, jj+1]
        py = _w00*Yr[ii, jj] + _w10*Yr[ii, jj+1] + _w01*Yr[ii+1, jj] + _w11*Yr[ii+1, jj+1]
        vals = _rgi(np.column_stack([py.ravel(), px.ravel()])).reshape(px.shape)
        with np.errstate(all="ignore"):
            _area[k:k+_chunk] = np.nanmean(vals, axis=1)
    zi[_wi, _wj] = _area
    _gaps = water_mask & ~np.isfinite(zi)
    if _gaps.any():
        zi[_gaps] = _nearest(xc[_gaps], yc[_gaps])
    print(f"  area mean with {n_sub}x{n_sub} sub-points per cell; "
          f"{(water_mask & (zi <= 0)).sum():,} water cells are land or intertidal in GEBCO")
elif method == "cell_mean":
    flat = pt_i[inside] * n_cols_c + pt_j[inside]
    sums = np.bincount(flat, weights=depths[inside], minlength=zi.size)
    cnts = np.bincount(flat, minlength=zi.size)
    cell_mean = np.where(cnts > 0, sums / np.maximum(cnts, 1), np.nan).reshape(zi.shape)
    has_pts = water_mask & np.isfinite(cell_mean)
    no_pts  = water_mask & ~np.isfinite(cell_mean)
    zi[has_pts] = cell_mean[has_pts]
    if no_pts.any():
        zi[no_pts] = _nearest(xc[no_pts], yc[no_pts])
    print(f"  {has_pts.sum():,} water cells averaged, {no_pts.sum():,} filled by nearest point")
elif method == "linear":
    zi[water_mask] = griddata((lons, lats), depths, (xc[water_mask], yc[water_mask]), method="linear")
    gaps = water_mask & ~np.isfinite(zi)
    if gaps.any():
        zi[gaps] = _nearest(xc[gaps], yc[gaps])
elif method == "nearest":
    zi[water_mask] = _nearest(xc[water_mask], yc[water_mask])
else:
    raise ValueError(f"Unknown interp_method: {interp_method!r}")

# ---------------------------------------------------------------- merge local chart / survey data (optional)
if chart_xyz is not None:
    from scipy.ndimage import distance_transform_edt
    _lines = [l.strip() for l in Path(chart_xyz).read_text().splitlines()]
    _rows = [l.split()[:3] for l in _lines if l and not l.startswith("<")]
    _c = np.array(_rows, dtype=float)
    c_lon, c_lat, c_dep = _c[:, 0], _c[:, 1], _c[:, 2] + chart_datum_below_msl
    # cell of each sounding (works for rotated grids)
    _ang = np.arctan2(Yr[0, -1] - Yr[0, 0], Xr[0, -1] - Xr[0, 0]); _ca, _sa = np.cos(_ang), np.sin(_ang)
    def _unrot(x, y):
        ddx, ddy = np.asarray(x) - Xr[0, 0], np.asarray(y) - Yr[0, 0]
        return ddx * _ca + ddy * _sa, -ddx * _sa + ddy * _ca
    _ue, _ = _unrot(Xr[0, :], Yr[0, :]); _, _ve = _unrot(Xr[:, 0], Yr[:, 0])
    _s1 = np.sign(_ue[-1] - _ue[0]) or 1.0; _s2 = np.sign(_ve[-1] - _ve[0]) or 1.0
    _pu, _pv = _unrot(c_lon, c_lat)
    cj = np.searchsorted(_s1 * _ue, _s1 * _pu, side="right") - 1
    ci = np.searchsorted(_s2 * _ve, _s2 * _pv, side="right") - 1
    _in = (ci >= 0) & (ci < n_rows_c) & (cj >= 0) & (cj < n_cols_c)
    _flat = ci[_in] * n_cols_c + cj[_in]
    _sum = np.bincount(_flat, weights=c_dep[_in], minlength=zi.size)
    _cnt = np.bincount(_flat, minlength=zi.size)
    chart = np.where(_cnt > 0, _sum / np.maximum(_cnt, 1), np.nan).reshape(zi.shape)
    has_chart = np.isfinite(chart) & water_mask
    from scipy.ndimage import uniform_filter
    # local sounding density: fraction of the WATER cells within chart_max_gap cells that have soundings
    _win = 2 * chart_max_gap + 1
    _n_s = uniform_filter(has_chart.astype(float), size=_win, mode="constant")
    _n_w = uniform_filter(water_mask.astype(float), size=_win, mode="constant")
    density = np.where(_n_w > 0, _n_s / np.maximum(_n_w, 1e-9), 0.0)
    conf = np.clip(density / chart_min_density, 0.0, 1.0)      # 1 = well covered, < 1 = sparse soundings
    # fill gaps between soundings only inside well-covered areas
    _gap = water_mask & ~has_chart & (conf >= 1.0)
    if _gap.any() and _in.sum() >= 3:
        chart[_gap] = griddata((c_lon[_in], c_lat[_in]), c_dep[_in], (xc[_gap], yc[_gap]), method="linear")
    chart_cov = water_mask & np.isfinite(chart)
    core = chart_cov & (conf >= 1.0)
    # transition zone: weight falls from 1 at well-covered cells to 0 over chart_blend cells
    halo = np.zeros(zi.shape)
    near = np.full(zi.shape, np.nan)
    if core.any():
        _d, (_ni, _nj) = distance_transform_edt(~core, return_indices=True)
        halo = np.clip(1.0 - _d / (chart_blend + 1.0), 0.0, 1.0)
        near = np.where(core[_ni, _nj], chart[_ni, _nj], np.nan)
    # cells with chart data: weight = max(local confidence, transition weight)
    _wc = np.where(chart_cov, np.maximum(conf, halo), 0.0)
    zi[chart_cov] = _wc[chart_cov] * chart[chart_cov] + (1.0 - _wc[chart_cov]) * zi[chart_cov]
    # cells without chart data next to well-covered areas: blend toward the nearest well-covered value
    _b = water_mask & ~chart_cov & (halo > 0) & np.isfinite(near)
    zi[_b] = halo[_b] * near[_b] + (1.0 - halo[_b]) * zi[_b]
    _sparse = chart_cov & (conf < 1.0)
    print(f"Chart data merged: {_in.sum():,} soundings in the grid (Z0 = {chart_datum_below_msl} m), "
          f"{has_chart.sum():,} cells with soundings; {core.sum():,} well-covered cells "
          f"(incl. {(_gap & chart_cov).sum():,} gap-filled), {_sparse.sum():,} sparse cells weighted "
          f"(mean weight {np.mean(_wc[_sparse]) if _sparse.any() else 0:.2f}), {_b.sum():,} blended")

# ---------------------------------------------------------------- Gaussian smoothing (optional)
# Mask land as NaN, filter, then renormalize by weight (no land bias)
if sigma and sigma > 0:
    depths_nan = np.where(water_mask, zi, np.nan)
    num_filt = gaussian_filter(np.nan_to_num(depths_nan), sigma=sigma)
    w_filt   = gaussian_filter(np.isfinite(depths_nan).astype(float), sigma=sigma)
    zi_smooth = np.full_like(num_filt, np.nan)
    valid = w_filt > 0
    zi_smooth[valid] = num_filt[valid] / w_filt[valid]
    zi = np.where(water_mask, zi_smooth, nodata_val)
else:
    zi = np.where(water_mask, zi, nodata_val)

# ---------------------------------------------------------------- remove isolated spikes and pits
# A water cell is changed only if it is deeper than ALL its water neighbours (pit) or shallower than
# ALL of them (spike) by more than max(spike_abs, spike_rel * neighbours' median), and has at least
# 5 water neighbours. Features two or more cells wide (channels, ridges, banks) are kept, because
# the cell has at least one neighbour of similar depth.
if despike:
    hp = np.pad(np.where(water_mask, zi, np.nan), 1, constant_values=np.nan)
    nb = np.stack([hp[1+di:hp.shape[0]-1+di, 1+dj:hp.shape[1]-1+dj]
                   for di in (-1, 0, 1) for dj in (-1, 0, 1) if (di, dj) != (0, 0)])
    n_nb = np.isfinite(nb).sum(axis=0)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        nb_med, nb_max, nb_min = np.nanmedian(nb, 0), np.nanmax(nb, 0), np.nanmin(nb, 0)
    thr = np.maximum(spike_abs, spike_rel * np.abs(nb_med))
    fix = water_mask & (n_nb >= 5) & (((zi - nb_max) > thr) | ((nb_min - zi) > thr))
    zi[fix] = nb_med[fix]
    print(f"  despike: {int(fix.sum()):,} isolated spike/pit cells replaced by their neighbours' median")

# ---------------------------------------------------------------- minimum depth
if min_depth is not None:
    zi[water_mask] = np.maximum(zi[water_mask], min_depth)

# ---------------------------------------------------------------- slope (rx0) limiting
_rx0_free = ~estuary_mask if estuary_mode == "no_rx0" else np.ones_like(water_mask)
def _pair_ok(a, b, fa=None, fb=None):
    ok = (a > 0) & (b > 0)                      # both water with positive depth (land is -99)
    if fa is not None:
        ok &= fa & fb                           # neither cell inside a "no_rx0" estuary
    if sigma_layer_depth is not None:
        ok &= np.minimum(a, b) < sigma_layer_depth
    return ok

def _rx0_pairs(h):
    """rx0 for horizontal and vertical neighbour pairs (NaN where not applicable)."""
    out = []
    f = _rx0_free
    for a, b, fa, fb in ((h[:, :-1], h[:, 1:], f[:, :-1], f[:, 1:]), (h[:-1, :], h[1:, :], f[:-1, :], f[1:, :])):
        with np.errstate(all="ignore"):
            out.append(np.where(_pair_ok(a, b, fa, fb), np.abs(a - b) / (a + b), np.nan))
    return out

def _rx0_max(h):
    rh, rv = _rx0_pairs(h)
    return np.nanmax(np.concatenate([rh.ravel(), rv.ravel(), [0.0]]))

if rx0_max is not None:
    rx0_before = _rx0_max(zi)
    h = zi.copy()
    n_iter = 0
    for n_iter in range(1, 2001):
        rh, rv = _rx0_pairs(h)
        if np.nanmax(np.concatenate([rh.ravel(), rv.ravel(), [0.0]])) <= rx0_max + 1e-6:
            break
        delta = np.zeros_like(h)
        # For a violating pair, move both depths toward each other (sum conserved) so rx0 = rx0_max.
        for r, (sa, sb) in ((rh, (np.s_[:, :-1], np.s_[:, 1:])), (rv, (np.s_[:-1, :], np.s_[1:, :]))):
            a, b = h[sa], h[sb]
            bad = np.nan_to_num(r) > rx0_max
            S = a + b
            deep_a = a > b
            target_deep, target_shal = S * (1 + rx0_max) / 2, S * (1 - rx0_max) / 2
            da = np.where(bad, np.where(deep_a, target_deep - a, target_shal - a), 0.0)
            db = np.where(bad, np.where(deep_a, target_shal - b, target_deep - b), 0.0)
            delta[sa] += 0.5 * da                      # relaxation: a cell may be in several pairs
            delta[sb] += 0.5 * db
        h += delta
    if min_depth is not None:
        h[water_mask] = np.maximum(h[water_mask], min_depth)
    _dh = np.where(water_mask, h - zi, 0.0)
    _changed = np.abs(_dh) > 0.01
    zi = h
    _where = f"h < {sigma_layer_depth:g} m" if sigma_layer_depth is not None else "all depths"
    print(f"  rx0 limiting ({_where}): max rx0 {rx0_before:.3f} -> {_rx0_max(zi):.3f} "
          f"in {n_iter} iterations; {_changed.sum():,} cells changed, "
          f"largest change {np.abs(_dh).max():.2f} m")

# ---------------------------------------------------------------- remove small water patches
labels, nfeat = label(water_mask)
for lbl in range(1, nfeat + 1):
    comp = (labels == lbl)
    if comp.sum() < min_patch:
        zi[comp] = nodata_val

print(f"\nBathymetric data interpolated")

# 6. Visualize and update depth values by clicking on the map
This code is currently efficient for not-so-large griddatas. To update large gridatas (e.g., 400 x 400 cells), convert first to shapefile, modify the cell depths in QGIS, and then convert the griddata to a MOHID-compatible file again. You can use this Jupyter Notebook to convert between formats.

In [ ]:
# -----------------------
# Setup and timing
# -----------------------
start_time = time.time()

# Assume x_grid, y_grid, and zi are defined externally
LonGrid = np.array(Xr)
LatGrid = np.array(Yr)
min_lon, max_lon = LonGrid.min(), LonGrid.max()
min_lat, max_lat = LatGrid.min(), LatGrid.max()

# Create a dedicated output widget for displaying interactive controls.
output = widgets.Output()
display(output)

# -----------------------
# Create the Map
# -----------------------
m = Map(center=(LatGrid.mean(), LonGrid.mean()), zoom=8)
marker = None  # For interactive marker

# Dictionary to hold block layers (for efficient redraw)
block_layers = {}
block_size = 10  # Adjust as needed

# Precompute cell corners (if LonGrid & LatGrid shape is (M, N), cells are (M-1, N-1)).
lon_sw = LonGrid[:-1, :-1]  # Southwest corner
lon_se = LonGrid[:-1,  1:]  # Southeast corner
lon_ne = LonGrid[1:,   1:]  # Northeast corner
lon_nw = LonGrid[1:,  :-1]  # Northwest corner

lat_sw = LatGrid[:-1, :-1]
lat_se = LatGrid[:-1,  1:]
lat_ne = LatGrid[1:,   1:]
lat_nw = LatGrid[1:,  :-1]

# -----------------------
# Color Mapping Globals and Functions
# -----------------------
_nbins = 10
_bins = None
_discrete_colors = None

def map_value_to_color(value):
    if value == -99:
        return "#ffffff00"  # Transparent for invalid cells
    # np.digitize returns indices in 1.._nbins+1; subtract 1 for 0-indexing.
    bin_index = np.digitize(value, _bins) - 1
    bin_index = int(np.clip(bin_index, 0, _nbins - 1))
    return _discrete_colors[bin_index]

def precompute_color_grid(zi, nbins=10):
    """
    Precompute a flat color mapping for the depth grid zi.
    Only valid cells (≠ -99) get a color from the viridis colormap.
    Also sets the global _bins, _nbins, and _discrete_colors.
    """
    global _bins, _nbins, _discrete_colors
    _nbins = nbins
    valid = zi != -99
    if np.any(valid):
        valid_min = zi[valid].min()
        valid_max = zi[valid].max()
    else:
        valid_min, valid_max = 0, 1
    _bins = np.linspace(valid_min, valid_max, nbins + 1)
    cmap = plt.colormaps.get_cmap('viridis')
    _discrete_colors = [mcolors.to_hex(c) for c in cmap(np.linspace(0, 1, nbins))]
    vectorized_map = np.vectorize(map_value_to_color)
    return vectorized_map(zi)

# Precompute colors for the entire grid (used as initial reference).
color_mapped_zi = precompute_color_grid(zi, nbins=_nbins)

# -----------------------
# GeoJSON Block Generation & Map Layer Updates
# -----------------------
def generate_block_geojson(block_row, block_col, block_size):
    """
    Create a GeoJSON FeatureCollection for cells within one spatial block.
    Only cells with valid depth (zi != -99) are included.
    """
    features = []
    total_rows, total_cols = zi.shape
    row_start = block_row * block_size
    row_end = min((block_row + 1) * block_size, total_rows)
    col_start = block_col * block_size
    col_end = min((block_col + 1) * block_size, total_cols)
    
    for i in range(row_start, row_end):
        for j in range(col_start, col_end):
            if zi[i, j] == -99:
                continue
            coordinates = [[
                [float(lon_sw[i, j]), float(lat_sw[i, j])],
                [float(lon_se[i, j]), float(lat_se[i, j])],
                [float(lon_ne[i, j]), float(lat_ne[i, j])],
                [float(lon_nw[i, j]), float(lat_nw[i, j])],
                [float(lon_sw[i, j]), float(lat_sw[i, j])]  # Close the ring
            ]]
            feature = {
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": coordinates
                },
                "properties": {
                    "fill": map_value_to_color(zi[i, j]),
                    "stroke": "#000000",
                    "fill-opacity": 0.5,
                    "stroke-width": 0.2,
                    "i": i,
                    "j": j
                }
            }
            features.append(feature)
    return {"type": "FeatureCollection", "features": features}

def update_all_blocks():
    """
    Clear and update all blocks by generating GeoJSON layers for the entire grid.
    """
    global block_layers
    for layer in block_layers.values():
        m.remove_layer(layer)
    block_layers = {}
    
    total_rows, total_cols = zi.shape
    n_block_rows = (total_rows + block_size - 1) // block_size
    n_block_cols = (total_cols + block_size - 1) // block_size
    
    for br in range(n_block_rows):
        for bc in range(n_block_cols):
            fc = generate_block_geojson(br, bc, block_size)
            if fc["features"]:
                layer = GeoJSON(
                    data=fc,
                    style_callback=lambda feature: {
                        "fillColor": feature["properties"]["fill"],
                        "color": feature["properties"]["stroke"],
                        "weight": feature["properties"]["stroke-width"],
                        "fillOpacity": feature["properties"]["fill-opacity"],
                    }
                )
                m.add_layer(layer)
                block_layers[(br, bc)] = layer

# For initial rendering, generate all blocks.
update_all_blocks()
print(f"Total time: {time.time() - start_time:.2f} seconds")
display(m)

# -----------------------
# Function to Compute Grid Cell from Clicked Coordinate
# -----------------------
def get_grid_index(lon, lat):
    """
    Convert clicked coordinates (lon, lat) into grid indices (j, i)
    by comparing the click location to cell centers.
    
    Returns:
       (j, i): j is the column index and i is the row index in the depth grid.
       Returns (None, None) if click is outside the grid.
    """
    if lon < min_lon or lon > max_lon or lat < min_lat or lat > max_lat:
        print("Warning: Click is outside grid boundaries.")
        return None, None

    centers_lon = 0.25 * (LonGrid[:-1, :-1] + LonGrid[:-1, 1:] + LonGrid[1:, :-1] + LonGrid[1:, 1:])
    centers_lat = 0.25 * (LatGrid[:-1, :-1] + LatGrid[:-1, 1:] + LatGrid[1:, :-1] + LatGrid[1:, 1:])
    distances = np.sqrt((centers_lon - lon) ** 2 + (centers_lat - lat) ** 2)
    flat_index = np.argmin(distances)
    i, j = np.unravel_index(flat_index, centers_lon.shape)
    return j, i  # (column index, row index)

# -----------------------
# Function to Update Only the Block that Contains a Changed Cell
# -----------------------
def update_block_for_cell(i, j):
    """
    Regenerate the GeoJSON layer for the block containing cell (i, j).
    """
    global block_layers
    block_row = i // block_size
    block_col = j // block_size
    fc = generate_block_geojson(block_row, block_col, block_size)
    key = (block_row, block_col)
    
    if key in block_layers:
        m.remove_layer(block_layers[key])
    
    if fc["features"]:
        new_layer = GeoJSON(
            data=fc,
            style_callback=lambda feature: {
                "fillColor": feature["properties"]["fill"],
                "color": feature["properties"]["stroke"],
                "weight": feature["properties"]["stroke-width"],
                "fillOpacity": feature["properties"]["fill-opacity"],
            }
        )
        m.add_layer(new_layer)
        block_layers[key] = new_layer
    else:
        if key in block_layers:
            del block_layers[key]

# -----------------------
# Interactive Cell Depth Update Callback
# -----------------------
def update_depth(**kwargs):
    """
    On a map click, display a text box and a submit button (inside our output widget)
    to let the user update the cell's depth. Once submitted, only the affected block is updated.
    """
    global marker
    if kwargs.get("type") != "click":
        return
    
    lat_click, lon_click = kwargs["coordinates"]
    if marker:
        m.remove_layer(marker)
    marker = Marker(location=(lat_click, lon_click))
    m.add_layer(marker)
    
    j, i = get_grid_index(lon_click, lat_click)
    if j is None or i is None:
        return
    
    # Create the text box and button using ipywidgets.
    input_box = widgets.Text(
        placeholder='Enter new depth value...',
        description=f'Cell ({i}, {j}) Depth {zi[i, j]:.1f}:',
        disabled=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    submit_button = widgets.Button(description="Submit")
    
    def on_submit(_):
        try:
            new_val = float(input_box.value)
            zi[i, j] = new_val
            update_block_for_cell(i, j)
        except ValueError:
            print("Invalid input. Enter a numeric value.")
    
    submit_button.on_click(on_submit)
    
    # Instead of a global clear_output that wipes the map, clear only our output widget.
    with output:
        output.clear_output(wait=True)
        display(input_box, submit_button)

# Attach the interactive callback for map clicks.
m.on_interaction(update_depth)

# 7. Save the griddata to a MOHID-compatible file

In [ ]:
rows, cols = zi.shape

np_x = np.array(Xr)
np_y = np.array(Yr)

# Get current date and time
now = datetime.now()

# Format the date and time
formatted_date_time = now.strftime("%d-%m-%Y %H:%M:%S")

output_file = "mohid_griddata.dat"
with open(output_file, "w") as f:
    f.write("PROJ4_STRING              : +proj=longlat +datum=WGS84 +no_defs\n")
    f.write("COMENT1                   : Grid generated by MOHID Jupyter Notebook\n")
    f.write("COMENT1                   : Generation Time: " + formatted_date_time + "\n")
    f.write("LATITUDE                  : " + str(y0) + "\n")
    f.write("LONGITUDE                 : " + str(x0) + "\n")
    f.write("COORD_TIP                 : 4\n")
    f.write("ILB_IUB                   : 1 " + str(int(rows)) + "\n")
    f.write("JLB_JUB                   : 1 " + str(int(cols)) + "\n")
    f.write("ORIGIN                    : " + str(x0) + " " + str(y0) + "\n")
    f.write("GRID_ANGLE                : " + str(grid_angle) + "\n")
    f.write("CONSTANT_SPACING_X        : 1\n")
    f.write("CONSTANT_SPACING_Y        : 1\n")
    f.write("DX                        : " + str(dx) + "\n")
    f.write("DY                        : " + str(dy) + "\n")
    f.write("FILL_VALUE                : -99\n")
    f.write("<BeginGridData2D>\n")
    
    for i in range(rows):
        for j in range(cols):
            f.write(f"{zi[i][j]:.2f}\n")
            
    f.write("<EndGridData2D>")       
    

print(f"\nGrid saved to {output_file}")

# 8. Convert the griddata to a shapefile

In [ ]:
# Create a shapefile writer object
w = shapefile.Writer('depth_grid_cells', shapefile.POLYGON)
w.field('i', 'N')       # Row index (i)
w.field('j', 'N')       # Column index (j)
w.field('Depth', 'F', decimal=2)  # Depth value (float)

# Loop through the grid and create polygon cells
for i in range(len(zi)):
    for j in range(len(zi[i])):
        depth = zi[i, j]  # Get depth value
        
        #if depth <= -99:
        #    depth = float('nan')
        
        # Define the 4 corner points of the grid cell with double precision
        lon1, lat1 = round(Xr[i, j], 15), round(Yr[i, j], 15)         # Bottom-left
        lon2, lat2 = round(Xr[i, j+1], 15), round(Yr[i, j+1], 15)     # Bottom-right
        lon3, lat3 = round(Xr[i+1, j+1], 15), round(Yr[i+1, j+1], 15) # Top-right
        lon4, lat4 = round(Xr[i+1, j], 15), round(Yr[i+1, j], 15) # Top-left

        # Create a polygon for the grid cell
        w.poly([[ (lon1, lat1), (lon2, lat2), (lon3, lat3), (lon4, lat4), (lon1, lat1) ]])  
        
        # Add attributes (Row, Col, Depth)
        w.record(i+1, j+1, depth)

# Save the shapefile
w.close()

print("Shapefile 'depth_grid_cells.shp' created successfully.")


# 9. Save shapefile to MOHID griddata

In [3]:
# ───────── USER CONFIG ─────────
shapefile_path = "depth_grid_cells.shp"
output_file    = "mohid_griddata_from_shapefile.dat"
fill_val       = -99.0

# 1. Read & inspect
gdf = gpd.read_file(shapefile_path)

# 2. Handle missing CRS by inferring WGS84 if bounds look geographic
if gdf.crs is None:
    minx, miny, maxx, maxy = gdf.total_bounds
    if -180 <= minx <= 180 and -90 <= miny <= 90:
        gdf.set_crs("EPSG:4326", inplace=True)
        print("No CRS: assuming EPSG:4326")
    else:
        raise ValueError("Shapefile has no CRS and bounds exceed geographic ranges.")

# 3. Reproject to WGS84 if needed
if gdf.crs.to_string() != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# 4. Flatten MultiPolygons
gdf.geometry = gdf.geometry.map(
    lambda geom: geom if geom.geom_type == "Polygon" else unary_union(geom)
)

# 5. Grid geometry from the cell indices (i = row, j = column), not from edge directions:
#    the direction from each cell to its neighbour in the next column gives the grid angle,
#    and the spacing between neighbouring cell centres gives dx and dy.
if not {"i", "j"}.issubset(gdf.columns):
    raise ValueError("The shapefile needs integer fields 'i' (row) and 'j' (column), as written by section 8.")
cen = pd.DataFrame({"i": gdf["i"].astype(int).to_numpy(), "j": gdf["j"].astype(int).to_numpy(),
                    "x": [p.centroid.x for p in gdf.geometry], "y": [p.centroid.y for p in gdf.geometry]})
nxt_col = cen.merge(cen, left_on=["i", cen["j"] + 1], right_on=["i", "j"], suffixes=("", "_n"))
nxt_row = cen.merge(cen, left_on=[cen["i"] + 1, "j"], right_on=["i", "j"], suffixes=("", "_n"))
if nxt_col.empty or nxt_row.empty:
    raise ValueError("Could not find neighbouring cells along both rows and columns.")
ux_vec = np.array([np.median(nxt_col["x_n"] - nxt_col["x"]), np.median(nxt_col["y_n"] - nxt_col["y"])])
uy_vec = np.array([np.median(nxt_row["x_n"] - nxt_row["x"]), np.median(nxt_row["y_n"] - nxt_row["y"])])
grid_angle = math.degrees(math.atan2(ux_vec[1], ux_vec[0]))   # counter-clockwise from east, as in section 3
grid_angle = 0.0 if abs(grid_angle) < 1e-6 else round(grid_angle, 6)
dx = float(np.hypot(*ux_vec))
dy = float(np.hypot(*uy_vec))

# 6. Origin = outer (south-west in grid terms) corner of cell (1, 1)
c11 = cen[(cen["i"] == cen["i"].min()) & (cen["j"] == cen["j"].min())]
if c11.empty:
    raise ValueError("Cell (1, 1) is missing from the shapefile.")
origin_x = float(c11["x"].iloc[0] - 0.5 * ux_vec[0] - 0.5 * uy_vec[0])
origin_y = float(c11["y"].iloc[0] - 0.5 * ux_vec[1] - 0.5 * uy_vec[1])

# 8. Build depth array
rows, cols = int(gdf["i"].max()), int(gdf["j"].max())
zi = np.full((rows, cols), fill_val, dtype=float)
for _, row in gdf.iterrows():
    i, j = int(row["i"]) - 1, int(row["j"]) - 1
    d = row.get("Depth", np.nan)
    zi[i, j] = float(d) if not np.isnan(d) else fill_val

# 9. Write MOHID .dat
now = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
with open(output_file, "w") as f:
    f.write("PROJ4_STRING              : +proj=longlat +datum=WGS84 +no_defs\n")
    f.write("COMMENT1                  : Grid generated by MOHID export\n")
    f.write(f"COMMENT2                  : Generated on {now}\n")
    f.write(f"LATITUDE                  : {origin_y}\n")
    f.write(f"LONGITUDE                 : {origin_x}\n")
    f.write("COORD_TIP                 : 4\n")
    f.write(f"ILB_IUB                   : 1 {rows}\n")
    f.write(f"JLB_JUB                   : 1 {cols}\n")
    f.write(f"ORIGIN                    : {origin_x} {origin_y}\n")
    f.write(f"GRID_ANGLE                : {grid_angle:.6f}\n")
    f.write("CONSTANT_SPACING_X        : 1\n")
    f.write("CONSTANT_SPACING_Y        : 1\n")
    f.write(f"DX                        : {dx:.10f}\n")
    f.write(f"DY                        : {dy:.10f}\n")
    f.write(f"FILL_VALUE                : {int(fill_val)}\n")
    f.write("<BeginGridData2D>\n")
    for ii in range(rows):
        for jj in range(cols):
            f.write(f"{zi[ii, jj]:.2f}\n")
    f.write("<EndGridData2D>\n")

print(f"Saved: {output_file} (angle={grid_angle:.4f}°, dx={dx:.10f}, dy={dy:.10f})")

No CRS: assuming EPSG:4326
Saved: mohid_griddata_from_shapefile.dat (angle=0.0000°, dx=0.0083330000, dy=0.0083330000)


# 10. Plot MOHID griddata

In [ ]:
np_x = np.array(Xr)
np_y = np.array(Yr)

x_min = np.min(np_x)
y_min = np.min(np_y)
x_max = np.max(np_x)
y_max = np.max(np_y)

#extent = [x_max, x_min, y_max, y_min]

# Calculate the cell spacing
dx = np.abs(np_x[0][1] - np_x[0][0])
dy = np.abs(np_y[1][0] - np_y[0][0])

# Expand the extent by n cells in all directions
n = 5

lon_min = x_min - n * dx
lon_max = x_max + n * dx
lat_min = y_min - n * dy
lat_max = y_max + n * dy

extent = [
    lon_min,  # Left
    lon_max,  # Right
    lat_min,  # Bottom
    lat_max]  # Top

def calculate_zoom_level():
    """
    Calculate zoom level based on the geographic extent.
    """
    # Approximate calculation for zoom level based on extent
    lat_range = lat_max - lat_min
    lon_range = lon_max - lon_min

    # Smaller ranges mean higher zoom levels
    range_avg = max(lat_range, lon_range)  # Focus on the larger dimension
    zoom = int(np.log2(360 / range_avg))   # Base-2 logarithm for zoom estimation

    # Limit zoom levels to reasonable values (e.g., 1 to 19)
    return max(1, min(zoom +1, 19))  

# Calculate zoom level
zoom_level = calculate_zoom_level()
print(f"Automatically calculated zoom level: {zoom_level}")

# Set the image size and create the figure
fig = plt.figure(figsize=(15, 15))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent(extent)

def image_spoof(self, tile):
    url = self._image_url(tile)  # Get the URL of the street map API
    req = Request(url)  # Start request
    req.add_header('User-agent', 'Anaconda 3')  # Add user agent to request
    fh = urlopen(req)
    im_data = io.BytesIO(fh.read())  # Get image
    fh.close()  # Close URL
    img = Image.open(im_data)  # Open image with PIL
    img = img.convert(self.desired_tile_form)  # Set image format
    return img, self.tileextent(tile), 'lower'  # Reformat for cartopy

cimgt.GoogleTiles.get_image = image_spoof  # Reformat web request for street map spoofing
osm_img = cimgt.GoogleTiles(style='satellite')
ax.add_image(osm_img, zoom_level) #you can increase the zoom level for better resolution of the satellite image

zi_mask = np.ma.masked_array(zi, zi <= -99)  # Mask to set transparency to a certain value

# Normalizing the data for the colorbar
norm = Normalize(vmin=np.min(zi_mask), vmax=np.max(zi_mask))

# Plot pcolormesh
pc = ax.pcolormesh(Xr, Yr, zi_mask, cmap='viridis', norm=norm)

# Plot scatter data if available
#if 'depths' in locals() and 'lons' in locals() and 'lats' in locals():
#    scatter = ax.scatter(lons, lats, c=depths, s=0.1, cmap='viridis', norm=norm, label='Data Points')

#Title
plt.title('Interpolated Bathymetric Data', fontsize=18, loc='center')

# Adjust the size of the colorbar
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1, axes_class=plt.Axes)

cbar = plt.colorbar(pc, cax=cax, orientation="vertical")
cbar.set_label('Depth (m)', labelpad=25, rotation=270, fontsize=16)
cbar.ax.tick_params(labelsize=14)

plt.savefig(rf'Griddata.png', format='png', dpi=300, bbox_inches='tight')